# Test de-identification code for 1000genomesVDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1755854301615_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-92-34.ap-southeast-1.compute.internal:34189
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
# source
trust_prefix = 's3://precise-trust/opendata-1000genomes/'

# input
vds_uri = trust_prefix + '1000genomes-vds-n3205.vds'

# output
hashed_vds_uri = trust_prefix + '1000genomes-vds-n3205.hashed.vds'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# read vds
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# describe reference_data
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [6]:
# check ref_block_max_length
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [7]:
# describe variant_data
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [8]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

In [9]:
# quick sanity
print("reference_data col key:", vds.reference_data.col_key)
print("variant_data  col key:", vds.variant_data.col_key)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

reference_data col key: <StructExpression of type struct{s: str}>
variant_data  col key: <StructExpression of type struct{s: str}>

In [ ]:
# create a vds_subset for testing
N_SAMPLES  = 10
N_VARIANTS = 1000

vref = vds.reference_data
vvar = vds.variant_data

# --- 1) choose first N_SAMPLES by current column order ---
first_samples = [c.s for c in vvar.cols().select().take(N_SAMPLES)]
samples_literal = hl.literal(set(first_samples))

vref_sub = vref.filter_cols(samples_literal.contains(vref.s))
vvar_sub = vvar.filter_cols(samples_literal.contains(vvar.s))

# --- 2) keep first N_VARIANTS from variant_data ---
vvar_sub = vvar_sub.head(N_VARIANTS)

# --- 3) align reference_data rows to loci present in variant_data ---
loci_ht = vvar_sub.rows().select().key_by("locus")  # set of loci kept
vref_sub = vref_sub.filter_rows(hl.is_defined(loci_ht[vref_sub.row_key]))

# --- 4) rebuild the VDS subset ---
vds_subset = hl.vds.VariantDataset(vref_sub, vvar_sub)

# (optional) quick sanity checks
print("subset ncols (ref, var):", vds_subset.reference_data.count_cols(), vds_subset.variant_data.count_cols())
print("subset nrows var:", vds_subset.variant_data.count_rows())
print("subset nrows ref:", vds_subset.reference_data.count_rows())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# replace original vds by vds_subset
vds = vds_subset

In [ ]:
# Preferred: set an environment variable before launching Jupyter:
#   export DEID_SALT="your-long-random-secret"
# SALT = os.getenv("DEID_SALT")

# For testing only, hardcode a fixed salt here:
SALT = "1234567890abcdef1234567890abcdef1234567890abcdef1234567890abcdef"

print("SALT (for test only):", SALT)


In [ ]:
# Make a per-sample token table from variant_data columns
# This table is keyed by 's' (current column key) and has the hashes we need.
ht_token = vds.variant_data.cols()

# Expecting fields:
#   ht_token.s (key)
#   ht_token.sample_info.npmid
#   ht_token.sample_info.pid

ht_token = ht_token.annotate(
    hash_npmid = hl.sha256(hl.str(SALT) + hl.str(ht_token.sample_info.npmid)),
    hash_pid   = hl.sha256(hl.str(SALT) + hl.str(ht_token.sample_info.pid)),
)

ht_token = ht_token.select("hash_npmid", "hash_pid")

ht_token.describe()
ht_token.show(5)

In [ ]:
# annotate and re-key variant_data
vd = vds.variant_data

# add hashed IDs as temporary cols
vd = vd.annotate_cols(
    _hash_npmid = ht_token[vd.col_key].hash_npmid,
    _hash_pid   = ht_token[vd.col_key].hash_pid,
)

# update sample_info with hashed values and set s to hashed_npmid
vd = vd.annotate_cols(
    sample_info = vd.sample_info.annotate(
        npmid = vd._hash_npmid,    # overwrite with hashed npmid
        pid   = vd._hash_pid,      # overwrite with hashed pid
    ),
    s = vd._hash_npmid,            # new visible sample ID = hashed npmid
).drop("_hash_npmid", "_hash_pid")

# rekey by the new s
vd = vd.key_cols_by(vd.s)

# Cell 4b: annotate and re-key reference_data to the same new 's'
rd = vds.reference_data
rd = rd.annotate_cols(
    s = ht_token[rd.col_key].hash_npmid
).key_cols_by(rd.s)

# Cell 4c: rebuild VDS
vds_deid = hl.vds.VariantDataset(rd, vd)


In [ ]:
# === sanity checks for a VDS after de-identification ===
vref = vds_deid.reference_data
vvar = vds_deid.variant_data

# 1) exact key-set equality between reference_data and variant_data
ref_keys = vref.cols().select()          # keyed by 's'
var_keys = vvar.cols().select()

missing_in_var = ref_keys.anti_join(var_keys).count()
missing_in_ref = var_keys.anti_join(ref_keys).count()

print(f"missing_in_variant_data: {missing_in_var}")
print(f"missing_in_reference_data: {missing_in_ref}")
assert missing_in_var == 0 and missing_in_ref == 0, \
    "Column key sets differ between reference_data and variant_data."

# (Optional but useful) still print counts
n_ref = vref.count_cols()
n_var = vvar.count_cols()
print(f"n cols (ref, var): {n_ref}, {n_var}")

# 2) enforce sha256-hex format for all keys (64 lowercase hex chars)
key_ht = vvar.cols()
bad_key_ct = key_ht.aggregate(
    hl.agg.count_where(~key_ht.s.matches(r'^[0-9a-f]{64}$'))
)
print("bad key format count:", bad_key_ct)
assert bad_key_ct == 0, "Some column keys are not sha256-hex."

# 3) confirm sample_info has hashed values:
#    - npmid should equal the column key (since you set s := hash(npmid))
#    - pid should also be a sha256-hex string
bad_npmid_mismatch = key_ht.aggregate(
    hl.agg.count_where(key_ht.sample_info.npmid != key_ht.s)
)
bad_pid_nonhex = key_ht.aggregate(
    hl.agg.count_where(~key_ht.sample_info.pid.matches(r'^[0-9a-f]{64}$'))
)

print("bad_npmid_mismatch:", bad_npmid_mismatch)
print("bad_pid_nonhex:", bad_pid_nonhex)
assert bad_npmid_mismatch == 0, "sample_info.npmid != column key for some samples."
assert bad_pid_nonhex == 0, "Some sample_info.pid are not sha256-hex."

# 4) quick peek (non-sensitive)
vvar.select_cols("sample_info").cols().show(5)

In [ ]:
# write output
vds_deid.write(hashed_vds_uri, overwrite=True)
print("Wrote:", hashed_vds_uri)
